# AraForge — Notebook 1: Ingestion & Processing (REVISED)
*Fixes applied: Whisper `large-v3` (was `large`); final audio standardized to 22.05 kHz / stereo / 16-bit PCM (matches the paper); the previously-incomplete transcription+segmentation+Excel-writing stage is now implemented end-to-end.*

# 1. Environment Setup & Dependencies

In [ ]:
# System
!apt-get install ffmpeg -y -q
# Python libs. NOTE: Spleeter pins TensorFlow; on modern Colab install it in an isolated step.
!pip install -q yt-dlp pydub pandas tqdm noisereduce speechbrain torchaudio openai-whisper openpyxl
# Spleeter is sensitive to TF versions; if separation fails, comment it out and rely on noisereduce only.
!pip install -q spleeter
print("Dependencies installed.")

# 2. Global Configuration & Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# ========================= PIPELINE CONFIG =========================
DETECT_EMOTIONS = False          # heavy; emotion is handled in Notebook 2
TARGET_SR       = 22050          # paper spec: 22.05 kHz
TARGET_CHANNELS = 2              # paper spec: stereo
WHISPER_MODEL   = "large-v3"     # paper spec: Whisper Large-v3 (was "large")
CHUNK_MIN       = 10             # minutes per memory-safe cleaning chunk
print(f"Init OK | Whisper={WHISPER_MODEL} | {TARGET_SR} Hz / {TARGET_CHANNELS}ch | emotions={DETECT_EMOTIONS}")

# 3. Phase 1: Audio Ingestion (YouTube or Drive)

In [ ]:
from yt_dlp import YoutubeDL
import shutil

def setup_folder_structure(main_folder, project_name):
    project_folder = os.path.join(main_folder, project_name)
    main_folder_path = os.path.join(project_folder, 'Main')
    parts_folder_path = os.path.join(project_folder, 'Parts')
    os.makedirs(main_folder_path, exist_ok=True)
    os.makedirs(parts_folder_path, exist_ok=True)
    return main_folder_path, parts_folder_path

def download_from_youtube(main_folder, urls, audio_format):
    if audio_format.lower() not in ["mp3", "wav"]:
        print("Invalid format; choose mp3 or wav."); return
    for url in urls.split(','):
        url = url.strip()
        try:
            with YoutubeDL({'quiet': True}) as ydl:
                info = ydl.extract_info(url, download=False)
            video_title = info.get('title', 'Unknown').replace('/', '_')
            video_id = info.get('id', 'Unknown')
            project_name = f"{video_title}_{video_id}"
            main_folder_path, _ = setup_folder_structure(main_folder, project_name)
            opts = {'format': 'bestaudio/best',
                    'outtmpl': os.path.join(main_folder_path, f'{video_title}.%(ext)s'),
                    'postprocessors': [{'key': 'FFmpegExtractAudio',
                                        'preferredcodec': audio_format.lower(),
                                        'preferredquality': '192'}],
                    'quiet': True}
            print(f"Downloading: {video_title} ...")
            with YoutubeDL(opts) as dl: dl.download([url])
            print(f"Done ({audio_format.upper()}).")
        except Exception as e:
            print(f"Failed {url}: {e}")

def copy_from_drive(main_folder, file_path):
    if not os.path.exists(file_path):
        print(f"Not found: {file_path}"); return
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.mp3', '.wav']:
        print(f"Unsupported '{ext}'; need .mp3/.wav"); return
    file_name = os.path.basename(file_path)
    project_name = os.path.splitext(file_name)[0].replace('/', '_')
    main_folder_path, _ = setup_folder_structure(main_folder, project_name)
    shutil.copy2(file_path, os.path.join(main_folder_path, file_name))
    print(f"Copied -> {os.path.join(main_folder_path, file_name)}")

print("Select input:  [1] YouTube   [2] Drive file")
choice = input("Enter 1 or 2: ").strip()
main_folder = input("Pipeline workspace folder: ").strip()
if choice == '1':
    download_from_youtube(main_folder, input("YouTube URLs (comma-sep): ").strip(), input("Format (mp3/wav): ").strip())
elif choice == '2':
    copy_from_drive(main_folder, input("Absolute Drive file path: ").strip())
else:
    print("Invalid choice.")

# 4. Phase 2: Memory-Chunked Noise & Music Removal
*FIX: output is force-standardized to 22.05 kHz / stereo / 16-bit PCM. Spleeter emits 44.1 kHz, so we resample on recombination; the old `bitrate="16k"` (which corrupted quality) is removed.*

In [ ]:
from pydub import AudioSegment
import noisereduce as nr
import numpy as np

def clean_master_audio(input_path, main_folder_path):
    print(f"Cleaning: {input_path}")
    temp_dir = os.path.join(main_folder_path, "temp_chunks"); os.makedirs(temp_dir, exist_ok=True)
    clean_master_path = os.path.join(main_folder_path, "clean_master.wav")
    audio = AudioSegment.from_file(input_path)
    chunk_ms = CHUNK_MIN * 60 * 1000
    chunks = [audio[i:i+chunk_ms] for i in range(0, len(audio), chunk_ms)]

    use_spleeter = True
    try:
        from spleeter.separator import Separator
        separator = Separator('spleeter:2stems')
    except Exception as e:
        print(f"Spleeter unavailable ({e}); falling back to noisereduce only.")
        use_spleeter = False

    cleaned = []
    for idx, chunk in enumerate(chunks):
        print(f"  chunk {idx+1}/{len(chunks)}")
        samples = np.array(chunk.get_array_of_samples())
        reduced = nr.reduce_noise(y=samples, sr=chunk.frame_rate)
        nr_audio = chunk._spawn(reduced.astype(np.int16).tobytes())
        nr_audio = nr_audio.set_channels(2).set_frame_rate(TARGET_SR)
        temp_chunk = os.path.join(temp_dir, f"chunk_{idx}.wav")
        nr_audio.export(temp_chunk, format="wav")
        if use_spleeter:
            separator.separate_to_file(temp_chunk, temp_dir)
            vocal = os.path.join(temp_dir, f"chunk_{idx}", "vocals.wav")
            seg = AudioSegment.from_wav(vocal) if os.path.exists(vocal) else AudioSegment.from_wav(temp_chunk)
        else:
            seg = AudioSegment.from_wav(temp_chunk)
        # enforce target spec per chunk (Spleeter returns 44.1k)
        cleaned.append(seg.set_frame_rate(TARGET_SR).set_channels(TARGET_CHANNELS))

    print("Recombining ...")
    final = cleaned[0]
    for c in cleaned[1:]: final += c
    final = final.set_frame_rate(TARGET_SR).set_channels(TARGET_CHANNELS).set_sample_width(2)  # 16-bit
    final.export(clean_master_path, format="wav")   # PCM 16-bit, no bad bitrate flag
    shutil.rmtree(temp_dir, ignore_errors=True)
    print(f"Clean master -> {clean_master_path}")
    return clean_master_path

# 5. Phase 3: Whisper Transcription & Audio Segmentation
*FIX: this stage is now complete. It builds sentence-level segments from Whisper segment timestamps, slices the clean master into `Parts/<n>.wav` (22.05 kHz/stereo/16-bit), and writes `final_transcriptions_mapping.xlsx` with the columns Notebook 2 expects.*

In [ ]:
import whisper
import pandas as pd
from tqdm import tqdm

model = whisper.load_model(WHISPER_MODEL)
parent_dir = main_folder

for sub_dir in os.listdir(parent_dir):
    sub_dir_path = os.path.join(parent_dir, sub_dir)
    if not os.path.isdir(sub_dir_path): continue
    main_folder_path = os.path.join(sub_dir_path, "Main")
    if not os.path.exists(main_folder_path): continue
    media = [f for f in os.listdir(main_folder_path) if f.lower().endswith((".mp3", ".wav"))]
    media = [f for f in media if f != "clean_master.wav"]
    if not media: continue

    raw_audio_file = os.path.join(main_folder_path, media[0])
    clean_audio_file = clean_master_audio(raw_audio_file, main_folder_path)

    output_dir = os.path.join(sub_dir_path, "Parts"); os.makedirs(output_dir, exist_ok=True)
    xlsx_path = os.path.join(sub_dir_path, "final_transcriptions_mapping.xlsx")

    print("Transcribing (Whisper large-v3) ...")
    result = model.transcribe(clean_audio_file, word_timestamps=True, verbose=False, language="ar")

    # ----- segmentation + slicing -----
    full = AudioSegment.from_file(clean_audio_file)
    rows = []
    for i, seg in enumerate(tqdm(result.get("segments", []), desc="Slicing"), start=1):
        text = (seg.get("text") or "").strip()
        if not text: continue
        start_ms = int(seg["start"] * 1000); end_ms = int(seg["end"] * 1000)
        clip = full[start_ms:end_ms].set_frame_rate(TARGET_SR).set_channels(TARGET_CHANNELS).set_sample_width(2)
        clip.export(os.path.join(output_dir, f"{i}.wav"), format="wav")
        rows.append({"Audio Clip Number": i, "Transcript": text,
                     "Start": round(seg["start"], 3), "End": round(seg["end"], 3)})

    pd.DataFrame(rows).to_excel(xlsx_path, index=False)
    print(f"Wrote {len(rows)} segments -> {xlsx_path}")

print("\nNotebook 1 complete. Proceed to Notebook 2.")